# nb9: Elemental Property Expansion

**Goal:** Extend the C6 baseline (R² = 0.643, 99-row LOO) by adding new elemental-property-based descriptors. Hard cap at **10 total features** because of the small dataset (99 compounds).

**Data source:** `element_properties.csv` (sibling to this notebook), with element symbol in column `Name` and ~30 numeric properties.

**Strategy (3 phases):**
1. **Phase A — C6 + 1.** Evaluate C6 plus every new candidate (one at a time). Rank by Δ R².
2. **Phase B — C6 + 2.** Take top 5 from Phase A, evaluate all 10 pairs.
3. **Phase C — C6 + 3 / +4.** If best C6+2 beats best C6+1, extend further. Hard stop at 10 total features.

**Why C6+1 instead of pure greedy from scratch:**
- C6 is already validated; pure greedy would waste cycles re-discovering known features.
- Risk: a new feature that would have been picked early in pure greedy may be masked by C6's existing coverage. We accept this for speed.
- If you want pure greedy later, swap `BASELINE_FEATURES` to `[]` and increase the per-step cap.

**Honest caveats** (matter for the report, not today):
- LOO scores from this kind of search are slightly optimistic (selection bias). True held-out R² is probably ~0.05 lower.
- We don't retune XGBoost per feature set. Final paper should do per-set tuning.
- Single seed (42). Final report = run with seeds [0..9] and report mean ± std.


## Cell 1: Imports & paths

In [2]:
import os
import warnings
import numpy as np
import pandas as pd
from time import time
from itertools import combinations

from pymatgen.core import Composition

from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import r2_score, mean_absolute_error
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')

# ============================================================================
# PATHS  (notebook lives in Keshav-DDP/new-descriptors/)
# ============================================================================
BASE_DIR = os.path.abspath(os.path.join('..'))   # -> Keshav-DDP/

OLD_CSV = os.path.join(BASE_DIR, 'k-path', 'old+new_nb6', 'rashba_206_all_descriptors_old.csv')
NEW_CSV = os.path.join(BASE_DIR, 'k-path', 'old+new_nb6', 'rashba_206_all_descriptors.csv')

# element_properties.csv lives next to this notebook
ELEM_CSV = os.path.join('.', 'element_properties.csv')

RESULTS_DIR = os.path.join('.', 'nb9_elemental-results')
os.makedirs(RESULTS_DIR, exist_ok=True)

print('OLD_CSV  :', OLD_CSV,  '  exists:', os.path.exists(OLD_CSV))
print('NEW_CSV  :', NEW_CSV,  '  exists:', os.path.exists(NEW_CSV))
print('ELEM_CSV :', ELEM_CSV, '  exists:', os.path.exists(ELEM_CSV))
print('RESULTS  :', RESULTS_DIR)


OLD_CSV  : c:\Users\AbCMS_Lab\Desktop\Keshav-DDP\k-path\old+new_nb6\rashba_206_all_descriptors_old.csv   exists: True
NEW_CSV  : c:\Users\AbCMS_Lab\Desktop\Keshav-DDP\k-path\old+new_nb6\rashba_206_all_descriptors.csv   exists: True
ELEM_CSV : .\element_properties.csv   exists: True
RESULTS  : .\nb9_elemental-results


## Cell 2: Load and merge the two descriptor CSVs (same as nb7)

We replicate nb7's merge: prefix overlapping columns with `old_` / `new_`, then collapse to 99 unique compounds (one row per uid, with max Rashba)
for the LOO eval.

In [3]:
df_old = pd.read_csv(OLD_CSV)
df_new = pd.read_csv(NEW_CSV)

print(f'Old CSV: {df_old.shape[0]} rows, {df_old.shape[1]} cols')
print(f'New CSV: {df_new.shape[0]} rows, {df_new.shape[1]} cols')

ID_COLS = ['Formula', 'uid', 'kpath', 'Rashba_parameter']
TARGET = 'Rashba_parameter'

df_merged = df_old[ID_COLS].copy()
old_features = [c for c in df_old.columns if c not in ID_COLS]
new_features = [c for c in df_new.columns if c not in ID_COLS]
overlap = set(old_features) & set(new_features)
print(f'Overlapping feature names (will be prefixed): {len(overlap)}')

for col in old_features:
    name = f'old_{col}' if col in overlap else col
    df_merged[name] = df_old[col].values
for col in new_features:
    name = f'new_{col}' if col in overlap else col
    df_merged[name] = df_new[col].values

print(f'Merged: {df_merged.shape[0]} rows, {df_merged.shape[1]} cols')

# Collapse to 99 compounds: keep row with max Rashba per uid
idx_max = df_merged.groupby('uid')[TARGET].idxmax()
df_99 = df_merged.loc[idx_max].reset_index(drop=True)
y_99 = df_99[TARGET].values
print(f'99-row df: {df_99.shape[0]} compounds, target [{y_99.min():.3f}, {y_99.max():.3f}]')


Old CSV: 205 rows, 1263 cols
New CSV: 205 rows, 134 cols
Overlapping feature names (will be prefixed): 27
Merged: 205 rows, 1393 cols
99-row df: 99 compounds, target [0.393, 4.804]


## Cell 3: Reproduce the C6 baseline (sanity check)

**This must print R² ≈ 0.643.** If it doesn't, something is different from nb7 and we should NOT proceed to expansion. Common reasons for mismatch:
- Different XGBoost version (XGBoost behavior across versions can shift R² by 0.01-0.02)
- Different sklearn version
- The CSVs have changed since nb7 was run

If you see 0.62-0.66 the model is fine. If you see 0.45 or 0.50 something is wrong with the merge or the column resolution — stop and debug.

In [4]:
# XGBoost params -- COPIED VERBATIM from nb7 Cell 12
XGB_REG_PARAMS = dict(
    n_estimators=100, max_depth=3, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=1.0, reg_lambda=1.0,
    random_state=42, verbosity=0,
)

# C6 features (winning 6-feature set from nb7 Cell 21 exhaustive search)
BASELINE_FEATURES_RAW = [
    'E_pfrac_VBM',
    'E_pfrac_CBM',
    'radius_mean',
    'pmid_afs_gauss_std',
    'kpath_angle_deg',
    'ehull',
]


def resolve_features(feat_list, df_cols):
    """Resolve feature names: try original, then old_, then new_ prefix."""
    cols = set(df_cols)
    resolved = []
    missing = []
    for f in feat_list:
        if f in cols:
            resolved.append(f)
        elif f'old_{f}' in cols:
            resolved.append(f'old_{f}')
        elif f'new_{f}' in cols:
            resolved.append(f'new_{f}')
        else:
            missing.append(f)
    if missing:
        print(f'  WARNING missing columns: {missing}')
    return resolved


def eval_reg_99(features, df=None, y=None):
    """LOO R2 / MAE with XGBoost regressor on 99-row df."""
    if df is None: df = df_99
    if y is None:  y = y_99
    X = df[features].fillna(0).values
    model = XGBRegressor(**XGB_REG_PARAMS)
    y_pred = cross_val_predict(model, X, y, cv=LeaveOneOut())
    return r2_score(y, y_pred), mean_absolute_error(y, y_pred)


BASELINE_FEATURES = resolve_features(BASELINE_FEATURES_RAW, df_99.columns)
print('Baseline (C6) resolved as:')
for f in BASELINE_FEATURES:
    print(f'  - {f}')

print()
t0 = time()
r2_base, mae_base = eval_reg_99(BASELINE_FEATURES)
print(f'Baseline LOO R2 = {r2_base:.4f}   MAE = {mae_base:.4f}   ({time()-t0:.1f}s)')
print(f'Expected from nb7: R2 ~ 0.643')

if abs(r2_base - 0.643) > 0.05:
    print('\n  WARNING: baseline R2 differs from nb7 by more than 0.05.')
    print('  Investigate before trusting Phase A/B/C results.')
else:
    print('\n  Baseline reproduces. Safe to proceed to expansion.')


Baseline (C6) resolved as:
  - old_E_pfrac_VBM
  - old_E_pfrac_CBM
  - old_radius_mean
  - pmid_afs_gauss_std
  - old_kpath_angle_deg
  - old_ehull

Baseline LOO R2 = 0.6434   MAE = 0.4287   (1.6s)
Expected from nb7: R2 ~ 0.643

  Baseline reproduces. Safe to proceed to expansion.


## Cell 4: Load `element_properties.csv` and build the per-compound feature matrix

For each compound (each row of `df_99`):
1. Parse the formula string to get elements + stoichiometric fractions.
2. Look up each element's properties from `element_properties.csv`.
3. Aggregate across the elements present using **5 aggregations**: weighted-mean (by stoichiometry), std, min, max, range.

So if there are P numeric properties in the CSV, we get **5 × P new candidate features**.

**Missing-data policy:** if any element in a compound has NaN for a given property, that property is skipped for that compound (the cell becomes NaN). XGBoost can handle NaN natively, but to keep parity with `eval_reg_99` (which does `.fillna(0)`), we should also be aware that 0 may be a misleading default. **For features where 0 is far outside the natural value range (e.g. ionization energy), consider median-impute instead of zero-fill.** I leave this as a TODO marked in code.

In [5]:
# Load element properties
elem_df = pd.read_csv(ELEM_CSV)
print(f'element_properties.csv: {elem_df.shape[0]} elements, {elem_df.shape[1]} columns')
print(f'Columns: {list(elem_df.columns)}')
print()
print(elem_df.head())

# The first column should be 'Name' (element symbol). Index by it for fast lookup.
assert 'Name' in elem_df.columns, "Expected 'Name' column with element symbols"
elem_df = elem_df.set_index('Name')

# Identify numeric property columns (skip any non-numeric ones)
numeric_props = [c for c in elem_df.columns if pd.api.types.is_numeric_dtype(elem_df[c])]
non_numeric = [c for c in elem_df.columns if c not in numeric_props]
print(f'\nNumeric properties ({len(numeric_props)}): {numeric_props}')
if non_numeric:
    print(f'Skipped non-numeric columns: {non_numeric}')

# Quick health check: how many NaNs per property?
nan_counts = elem_df[numeric_props].isna().sum()
print(f'\nProperties with NaN values:')
for p, n in nan_counts.items():
    if n > 0:
        print(f'  {p}: {n} elements missing')


element_properties.csv: 83 elements, 32 columns
Columns: ['Name', 'Z', 'M', 'G', 'IEI', 'IEII', 'EA', 'ChiP', 'ChiA', 'Rvdw', 'Rc', 'Ra', 'Rpp_s', 'Rpp_p', 'Rpp_d', 'MP', 'BP', 'Rho', 'MV', 'Hf', 'Hv', 'Kappa', 'CvM', 'B', 'MendeleevNo', 'PoissonsRatio', 'Kbulk', 'YoungsModulus', 'CoefficientOfLinearThermalExpansion', 'AverageCationicRadius', 'AverageAnionicRadius', 'Energy']

  Name     Z           M     G       IEI      IEII     EA  ChiP   ChiA  Rvdw  \
0   Ag  47.0  107.868200  11.0  7.576230  21.47746  125.6  1.93  1.870  2.11   
1   Al  13.0   26.981539  13.0  5.985768  18.82855   42.5  1.61  1.613  1.84   
2   As  33.0   74.921600  15.0  9.788600  18.58920   78.0  2.18  2.211  1.85   
3   Au  79.0  196.966569  11.0  9.225530  20.20000  222.8  2.54    NaN  2.14   
4    B   5.0   10.811000  13.0  8.298020  25.15480   26.7  2.04  2.051  1.92   

   ...    CvM  B  MendeleevNo  PoissonsRatio  Kbulk  YoungsModulus  \
0  ...  0.235  3         71.0           0.37  100.0           83.0   

In [7]:
# DIAGNOSTIC: which row(s) have a bad formula?
print('Formula health check:')
print(f'  Total rows: {len(df_99)}')
print(f'  NaN formulas: {df_99["Formula"].isna().sum()}')
print(f'  Empty/whitespace formulas: {(df_99["Formula"].fillna("").str.strip() == "").sum()}')
bad_mask = df_99['Formula'].fillna('').str.strip() == ''
bad = df_99[bad_mask]
if len(bad):
    print(f'\nBad rows ({len(bad)}):')
    print(bad[['Formula', 'uid']].to_string())

Formula health check:
  Total rows: 99
  NaN formulas: 0
  Empty/whitespace formulas: 1

Bad rows (1):
   Formula           uid
76          aa9a981d89aa


In [8]:
def safe_get_composition_weights(row):
    """Robust formula parser: tries Formula, falls back to uid prefix.
    Returns dict or None if both fail."""
    for source in ['Formula', 'uid']:
        raw = row.get(source, '')
        if not isinstance(raw, str):
            continue
        # Clean: strip whitespace; for uid, take part before any '-' or digits
        s = raw.strip()
        if source == 'uid':
            # C2DB uids look like 'BiTeI-abc123' or 'MoS2-xyz'; take part before '-'
            s = s.split('-')[0]
        if not s:
            continue
        try:
            comp = Composition(s)
            total = sum(comp.values())
            if total > 0:
                return {el.symbol: amt / total for el, amt in comp.items()}
        except Exception:
            continue
    return None


# Build feature matrix: one row per compound, columns = {prop}_{agg}
new_feature_rows = []
missing_elem_warnings = set()
n_skipped = 0

for idx, row in df_99.iterrows():
    weights_dict = safe_get_composition_weights(row)

    if weights_dict is None:
        # Both Formula and uid failed -- fill row with NaN
        n_skipped += 1
        print(f'  Row {idx}: could not parse Formula={row.get("Formula")!r} or uid={row.get("uid")!r}')
        feat_row = {}
        for prop in numeric_props:
            for agg in ['w_mean', 'std', 'min', 'max', 'range']:
                feat_row[f'el_{prop}_{agg}'] = np.nan
        new_feature_rows.append(feat_row)
        continue

    symbols = list(weights_dict.keys())
    weights = [weights_dict[s] for s in symbols]

    try:
        sub = elem_df.loc[symbols]
    except KeyError as e:
        missing_elem_warnings.add(str(e))
        feat_row = {}
        for prop in numeric_props:
            for agg in ['w_mean', 'std', 'min', 'max', 'range']:
                feat_row[f'el_{prop}_{agg}'] = np.nan
        new_feature_rows.append(feat_row)
        continue

    feat_row = {}
    for prop in numeric_props:
        agg = aggregate_property(symbols, weights, sub[prop].tolist())
        for k, v in agg.items():
            feat_row[f'el_{prop}_{k}'] = v
    new_feature_rows.append(feat_row)

elem_features_df = pd.DataFrame(new_feature_rows)
print(f'\nNew elemental feature matrix: {elem_features_df.shape}')
print(f'Total candidate features: {elem_features_df.shape[1]}')
print(f'Rows with NaN-filled features (parse failures): {n_skipped}')

if missing_elem_warnings:
    print(f'\n  Some elements missing from element_properties.csv: {missing_elem_warnings}')

# Drop columns that are all-NaN or constant (no information)
n_before = elem_features_df.shape[1]
nunique = elem_features_df.nunique(dropna=True)
keep_cols = [c for c in elem_features_df.columns if nunique[c] > 1]
elem_features_df = elem_features_df[keep_cols]
print(f'After dropping constant/all-NaN columns: {elem_features_df.shape[1]} (removed {n_before - elem_features_df.shape[1]})')

nan_cols = [c for c in elem_features_df.columns if elem_features_df[c].isna().any()]
print(f'Columns with at least one NaN: {len(nan_cols)}')

  Row 76: could not parse Formula=' ' or uid='aa9a981d89aa'

New elemental feature matrix: (99, 155)
Total candidate features: 155
Rows with NaN-filled features (parse failures): 1
After dropping constant/all-NaN columns: 148 (removed 7)
Columns with at least one NaN: 148


## Cell 5: Attach new features to df_99

Concat the new elemental feature columns to `df_99`. After this, every column we want for Phase A is in one DataFrame.

In [9]:
# Sanity: row counts must match
assert len(elem_features_df) == len(df_99), "row count mismatch between elem features and df_99"

df_99_ext = pd.concat([df_99.reset_index(drop=True), elem_features_df.reset_index(drop=True)], axis=1)
print(f'Extended df_99: {df_99_ext.shape[0]} rows, {df_99_ext.shape[1]} cols')

# All new candidate features
NEW_CANDIDATES = list(elem_features_df.columns)
print(f'\nNumber of new candidates to test in Phase A: {len(NEW_CANDIDATES)}')
print(f'Sample candidates: {NEW_CANDIDATES[:8]}')

# Re-resolve baseline against extended df (column names didn't change, but be safe)
BASELINE = resolve_features(BASELINE_FEATURES_RAW, df_99_ext.columns)


Extended df_99: 99 rows, 1541 cols

Number of new candidates to test in Phase A: 148
Sample candidates: ['el_Z_w_mean', 'el_Z_std', 'el_Z_min', 'el_Z_max', 'el_Z_range', 'el_M_w_mean', 'el_M_std', 'el_M_min']


## Cell 6: Phase A — C6 + 1 evaluation

For every candidate, train XGBoost on `BASELINE + [candidate]` and record LOO R². Sort by Δ R² vs baseline.

**Time estimate:** ~150 candidates × ~2s per LOO eval ≈ 5 minutes. Progress prints every 20 candidates.

In [10]:
print('=' * 70)
print('  PHASE A: C6 + 1 evaluation')
print('=' * 70)
print(f'  Baseline R2: {r2_base:.4f}')
print(f'  Candidates : {len(NEW_CANDIDATES)}')
print()

phase_a_results = []
t_phase = time()

for i, cand in enumerate(NEW_CANDIDATES):
    feats = BASELINE + [cand]
    try:
        r2, mae = eval_reg_99(feats, df=df_99_ext, y=y_99)
    except Exception as e:
        print(f'  [{i+1}/{len(NEW_CANDIDATES)}] {cand}: FAILED ({e})')
        continue

    phase_a_results.append({
        'candidate': cand,
        'r2': r2,
        'mae': mae,
        'delta_r2': r2 - r2_base,
    })

    if (i + 1) % 20 == 0 or i == len(NEW_CANDIDATES) - 1:
        elapsed = time() - t_phase
        remaining = elapsed / (i + 1) * (len(NEW_CANDIDATES) - i - 1)
        print(f'  [{i+1}/{len(NEW_CANDIDATES)}] elapsed {elapsed:.0f}s, eta {remaining:.0f}s')

phase_a_df = pd.DataFrame(phase_a_results).sort_values('delta_r2', ascending=False).reset_index(drop=True)

print()
print(f'Phase A complete in {time() - t_phase:.0f}s.')
print()
print('Top 15 candidates (C6 + 1):')
print(phase_a_df.head(15).to_string(index=False))

phase_a_df.to_csv(os.path.join(RESULTS_DIR, 'phase_a_ranking.csv'), index=False)
print(f'\nSaved: {os.path.join(RESULTS_DIR, "phase_a_ranking.csv")}')


  PHASE A: C6 + 1 evaluation
  Baseline R2: 0.6434
  Candidates : 148

  [20/148] elapsed 44s, eta 282s
  [40/148] elapsed 89s, eta 241s
  [60/148] elapsed 133s, eta 195s
  [80/148] elapsed 178s, eta 151s
  [100/148] elapsed 222s, eta 107s
  [120/148] elapsed 267s, eta 62s
  [140/148] elapsed 311s, eta 18s
  [148/148] elapsed 329s, eta 0s

Phase A complete in 329s.

Top 15 candidates (C6 + 1):
           candidate       r2      mae  delta_r2
        el_Rpp_d_max 0.640206 0.426156 -0.003177
      el_Rpp_d_range 0.632333 0.434717 -0.011050
           el_EA_std 0.630021 0.431663 -0.013362
        el_Rpp_d_std 0.628678 0.437780 -0.014705
         el_IEII_min 0.627210 0.428103 -0.016173
         el_ChiP_std 0.626823 0.425866 -0.016559
         el_Rvdw_max 0.624850 0.427955 -0.018533
         el_MP_range 0.624337 0.421077 -0.019046
           el_BP_max 0.623198 0.416974 -0.020185
        el_Hv_w_mean 0.623173 0.415454 -0.020210
         el_M_w_mean 0.622659 0.423817 -0.020724
  el_MendeleevN

## Cell 7: Phase B — C6 + 2 from top 5 of Phase A

Take top 5 candidates from Phase A, evaluate all `C(5,2) = 10` pairs.

In [11]:
print('=' * 70)
print('  PHASE B: C6 + 2 from top 5 of Phase A')
print('=' * 70)

TOP_K_FOR_PHASE_B = 5
top_candidates = phase_a_df.head(TOP_K_FOR_PHASE_B)['candidate'].tolist()
print(f'Top {TOP_K_FOR_PHASE_B} from Phase A:')
for i, c in enumerate(top_candidates, 1):
    print(f'  {i}. {c}  (Δ R2 = {phase_a_df.iloc[i-1]["delta_r2"]:+.4f})')

phase_b_results = []
for c1, c2 in combinations(top_candidates, 2):
    feats = BASELINE + [c1, c2]
    r2, mae = eval_reg_99(feats, df=df_99_ext, y=y_99)
    phase_b_results.append({
        'cand_1': c1,
        'cand_2': c2,
        'r2': r2,
        'mae': mae,
        'delta_r2': r2 - r2_base,
    })

phase_b_df = pd.DataFrame(phase_b_results).sort_values('delta_r2', ascending=False).reset_index(drop=True)

print('\nAll C6 + 2 pairs (sorted):')
print(phase_b_df.to_string(index=False))
phase_b_df.to_csv(os.path.join(RESULTS_DIR, 'phase_b_ranking.csv'), index=False)

# Best pair
best_a_r2 = phase_a_df.iloc[0]['r2']
best_b_r2 = phase_b_df.iloc[0]['r2']
print()
print(f'Best Phase A (C6 + 1): R2 = {best_a_r2:.4f}')
print(f'Best Phase B (C6 + 2): R2 = {best_b_r2:.4f}')
if best_b_r2 > best_a_r2:
    print(f'  -> +2 features helped: gain {best_b_r2 - best_a_r2:+.4f}')
else:
    print(f'  -> +2 features did NOT help over +1. Likely stop here.')


  PHASE B: C6 + 2 from top 5 of Phase A
Top 5 from Phase A:
  1. el_Rpp_d_max  (Δ R2 = -0.0032)
  2. el_Rpp_d_range  (Δ R2 = -0.0111)
  3. el_EA_std  (Δ R2 = -0.0134)
  4. el_Rpp_d_std  (Δ R2 = -0.0147)
  5. el_IEII_min  (Δ R2 = -0.0162)

All C6 + 2 pairs (sorted):
        cand_1         cand_2       r2      mae  delta_r2
  el_Rpp_d_max el_Rpp_d_range 0.642236 0.443872 -0.001147
  el_Rpp_d_std    el_IEII_min 0.640441 0.442598 -0.002942
  el_Rpp_d_max   el_Rpp_d_std 0.639111 0.445989 -0.004271
el_Rpp_d_range    el_IEII_min 0.638930 0.443941 -0.004452
el_Rpp_d_range   el_Rpp_d_std 0.635025 0.450692 -0.008358
el_Rpp_d_range      el_EA_std 0.626964 0.449383 -0.016419
  el_Rpp_d_max    el_IEII_min 0.624117 0.451290 -0.019265
     el_EA_std   el_Rpp_d_std 0.618030 0.450532 -0.025353
  el_Rpp_d_max      el_EA_std 0.598798 0.459910 -0.044585
     el_EA_std    el_IEII_min 0.596991 0.458181 -0.046392

Best Phase A (C6 + 1): R2 = 0.6402
Best Phase B (C6 + 2): R2 = 0.6422
  -> +2 features helped: 

## Cell 8: Phase C — C6 + 3 (only if Phase B beat Phase A)

If Phase B's best is better than Phase A's best, try adding a 3rd from the top-5 pool. We test all `C(5,3) = 10` triples.

Hard cap: total features ≤ 10. C6 + 3 = 9, so we have headroom for one more (Phase D, conditional).

In [12]:
best_a_r2 = phase_a_df.iloc[0]['r2']
best_b_r2 = phase_b_df.iloc[0]['r2']

if best_b_r2 <= best_a_r2:
    print('Phase B did not beat Phase A. Skipping Phase C (no evidence more features help).')
    phase_c_df = pd.DataFrame()
else:
    print('=' * 70)
    print('  PHASE C: C6 + 3 from top 5')
    print('=' * 70)
    phase_c_results = []
    for c1, c2, c3 in combinations(top_candidates, 3):
        feats = BASELINE + [c1, c2, c3]
        r2, mae = eval_reg_99(feats, df=df_99_ext, y=y_99)
        phase_c_results.append({
            'cand_1': c1, 'cand_2': c2, 'cand_3': c3,
            'r2': r2, 'mae': mae,
            'delta_r2': r2 - r2_base,
        })
    phase_c_df = pd.DataFrame(phase_c_results).sort_values('delta_r2', ascending=False).reset_index(drop=True)
    print('All C6 + 3 triples (sorted):')
    print(phase_c_df.to_string(index=False))
    phase_c_df.to_csv(os.path.join(RESULTS_DIR, 'phase_c_ranking.csv'), index=False)

    best_c_r2 = phase_c_df.iloc[0]['r2']
    print()
    print(f'Best Phase A: R2 = {best_a_r2:.4f}')
    print(f'Best Phase B: R2 = {best_b_r2:.4f}')
    print(f'Best Phase C: R2 = {best_c_r2:.4f}')


  PHASE C: C6 + 3 from top 5
All C6 + 3 triples (sorted):
        cand_1         cand_2       cand_3       r2      mae  delta_r2
  el_Rpp_d_max el_Rpp_d_range    el_EA_std 0.630534 0.450625 -0.012849
  el_Rpp_d_max el_Rpp_d_range el_Rpp_d_std 0.628754 0.453146 -0.014629
el_Rpp_d_range   el_Rpp_d_std  el_IEII_min 0.624910 0.445188 -0.018473
  el_Rpp_d_max   el_Rpp_d_std  el_IEII_min 0.622648 0.443887 -0.020735
  el_Rpp_d_max el_Rpp_d_range  el_IEII_min 0.621265 0.445304 -0.022117
     el_EA_std   el_Rpp_d_std  el_IEII_min 0.621001 0.456778 -0.022381
  el_Rpp_d_max      el_EA_std el_Rpp_d_std 0.620823 0.455855 -0.022560
el_Rpp_d_range      el_EA_std  el_IEII_min 0.618543 0.448231 -0.024840
  el_Rpp_d_max      el_EA_std  el_IEII_min 0.611957 0.451760 -0.031425
el_Rpp_d_range      el_EA_std el_Rpp_d_std 0.606028 0.456013 -0.037354

Best Phase A: R2 = 0.6402
Best Phase B: R2 = 0.6422
Best Phase C: R2 = 0.6305


## Cell 9: Final summary

Identify the best feature set across all phases and the path to it. Saves a single summary CSV.

In [13]:
print('=' * 70)
print('  FINAL SUMMARY')
print('=' * 70)
print(f'Baseline (C6, R2 = {r2_base:.4f})')
print()

candidates_summary = [
    ('Phase A (C6 + 1)', phase_a_df.iloc[0]['r2'] if len(phase_a_df) else None,
     [phase_a_df.iloc[0]['candidate']] if len(phase_a_df) else []),
    ('Phase B (C6 + 2)', phase_b_df.iloc[0]['r2'] if len(phase_b_df) else None,
     [phase_b_df.iloc[0]['cand_1'], phase_b_df.iloc[0]['cand_2']] if len(phase_b_df) else []),
]
if len(phase_c_df):
    candidates_summary.append(
        ('Phase C (C6 + 3)', phase_c_df.iloc[0]['r2'],
         [phase_c_df.iloc[0]['cand_1'], phase_c_df.iloc[0]['cand_2'], phase_c_df.iloc[0]['cand_3']])
    )

for name, r2, extras in candidates_summary:
    if r2 is None: continue
    delta = r2 - r2_base
    print(f'  {name}: R2 = {r2:.4f}  ({delta:+.4f})')
    print(f'    extras: {extras}')

# Best overall
best_phase, best_r2, best_extras = max(
    [(n, r, e) for n, r, e in candidates_summary if r is not None],
    key=lambda t: t[1]
)
print()
print(f'BEST: {best_phase}')
print(f'  R2 = {best_r2:.4f}  (Δ = {best_r2 - r2_base:+.4f})')
print(f'  Full feature set ({len(BASELINE) + len(best_extras)} features):')
for f in BASELINE + best_extras:
    print(f'    - {f}')

# Save summary
summary = pd.DataFrame([
    {'phase': name, 'r2': r2, 'delta_r2': r2 - r2_base if r2 is not None else None,
     'extras': ' + '.join(extras), 'n_features': len(BASELINE) + len(extras)}
    for name, r2, extras in candidates_summary if r2 is not None
])
summary.to_csv(os.path.join(RESULTS_DIR, 'final_summary.csv'), index=False)
print(f'\nSaved: {os.path.join(RESULTS_DIR, "final_summary.csv")}')


  FINAL SUMMARY
Baseline (C6, R2 = 0.6434)

  Phase A (C6 + 1): R2 = 0.6402  (-0.0032)
    extras: ['el_Rpp_d_max']
  Phase B (C6 + 2): R2 = 0.6422  (-0.0011)
    extras: ['el_Rpp_d_max', 'el_Rpp_d_range']
  Phase C (C6 + 3): R2 = 0.6305  (-0.0128)
    extras: ['el_Rpp_d_max', 'el_Rpp_d_range', 'el_EA_std']

BEST: Phase B (C6 + 2)
  R2 = 0.6422  (Δ = -0.0011)
  Full feature set (8 features):
    - old_E_pfrac_VBM
    - old_E_pfrac_CBM
    - old_radius_mean
    - pmid_afs_gauss_std
    - old_kpath_angle_deg
    - old_ehull
    - el_Rpp_d_max
    - el_Rpp_d_range

Saved: .\nb9_elemental-results\final_summary.csv


## Cell 10: (Optional) Inspect feature importance for the best set

Quick gut check: do the new features actually contribute, or is XGBoost ignoring them and the R² gain is noise?

In [ ]:
best_features = BASELINE + best_extras
X_best = df_99_ext[best_features].fillna(0).values
model = XGBRegressor(**XGB_REG_PARAMS)
model.fit(X_best, y_99)
imp = pd.Series(model.feature_importances_, index=best_features).sort_values(ascending=False)
print('Feature importance (gain) for best set:')
print(imp.to_string())

# Save importance
imp.to_csv(os.path.join(RESULTS_DIR, 'best_feature_importance.csv'), header=['importance'])


## Notes for next steps

1. If a new elemental feature shows up strongly (Δ R² > 0.02), check whether it's just a re-encoding of `radius_mean` or `max_Z4`. Compute pairwise correlation in df_99_ext.
2. **None of this is hyperparameter-tuned.** If we settle on a final 7-8 feature set for the report, run a quick GridSearchCV (n_estimators × max_depth × learning_rate) — typically gains another 0.01-0.03.
3. **Multi-seed runs.** For your paper, the result should be `R² = X.XXX ± Y.YYY` with seeds [0..9].
4. **Held-out test set.** The honest version of all this is: hold out ~20% before any feature search, do the search on the remaining 80%, report final number on the held-out. That's the pipeline a reviewer will ask for.
